# RAPID Hydrograph Analysis

Use this notebook after `RAPID/run_rapid_experiment.py` has finished. It reads the per-state outlet hydrographs and hydrograph metrics written under each state's `rapid/run/` folder.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [2]:
experiment_dir = Path("/Users/6256481/Code/river-hierarchy/network_variants/output/sarl03_indep")
run_registry = pd.read_csv(experiment_dir / "rapid_run_registry.csv")
run_registry.head()

,state_id,rapid_prep_dir,rapid_run_dir,qout_nc,status,hydrograph_status,outlet_hydrograph_csv,hydrograph_metrics_csv,hydrograph_metrics_json,event_start_source,...,outlet_excess_volume_m3,outlet_reach_count,outlet_reach_ids,hydrograph_duration_seconds,metric_config,fall_time_seconds,fall_time_50_seconds,fall_time_10_seconds,e_folding_time_seconds,rise_rate_cms_per_hour
0,S001_unit_5,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,ran,computed,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,auto_input_min_window,...,1.406658e+09,1,[796],948600.0,"{""event_start_time"": null, ""event_start_window...",NaN,303891.552642,NaN,NaN,17.333226
1,S002_unit_7,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,ran,computed,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,auto_input_min_window,...,1.408254e+09,1,[798],948600.0,"{""event_start_time"": null, ""event_start_window...",NaN,304385.098940,NaN,NaN,17.391797
2,S003_unit_20,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,ran,computed,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,auto_input_min_window,...,1.405985e+09,1,[797],948600.0,"{""event_start_time"": null, ""event_start_window...",NaN,304522.507999,NaN,NaN,17.328838
3,S004_unit_8,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,ran,computed,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,auto_input_min_window,...,1.406690e+09,1,[795],948600.0,"{""event_start_time"": null, ""event_start_window...",NaN,303893.259787,NaN,NaN,17.331846
4,S005_unit_12,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,ran,computed,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,auto_input_min_window,...,1.407362e+09,1,[796],948600.0,"{""event_start_time"": null, ""event_start_window...",NaN,303326.449626,NaN,NaN,17.333014


In [ ]:
metric_cols = [
    "state_id",
    "event_start_time_utc",
    "peak_time_utc",
    "event_end_time_utc",
    "peak_discharge_cms",
    "peak_excess_cms",
    "peak_excess_to_end_baseline_cms",
    "time_to_peak_seconds",
    "event_duration_seconds",
    "fall_time_seconds",
    "fall_time_50_seconds",
    "fall_time_10_seconds",
    "e_folding_time_seconds",
    "lag_to_inflow_peak_seconds",
    "peak_attenuation_ratio",
]
run_registry[metric_cols].sort_values("peak_discharge_cms", ascending=False).head(10)

In [ ]:
def load_state_hydrograph(state_id: str) -> pd.DataFrame:
    row = run_registry.loc[run_registry["state_id"].eq(state_id)].iloc[0]
    hydro = pd.read_csv(row["outlet_hydrograph_csv"])
    hydro["time_utc"] = pd.to_datetime(hydro["time_utc"], utc=True)
    return hydro

state_id = run_registry.loc[run_registry["status"].eq("ran"), "state_id"].iloc[0]
hydro = load_state_hydrograph(state_id)
hydro.head()

In [ ]:
row = run_registry.loc[run_registry["state_id"].eq(state_id)].iloc[0]
fig = go.Figure()
fig.add_trace(go.Scatter(x=hydro["time_utc"], y=hydro["q_inflow_cms"], mode="lines", name="Inflow"))
fig.add_trace(go.Scatter(x=hydro["time_utc"], y=hydro["q_outlet_cms"], mode="lines", name="Outlet"))
fig.add_vline(x=pd.to_datetime(row["event_start_time_utc"], utc=True), line_dash="dash", line_color="gray")
fig.add_vline(x=pd.to_datetime(row["peak_time_utc"], utc=True), line_dash="dot", line_color="red")
fig.update_layout(title=f"{state_id} outlet hydrograph", xaxis_title="time", yaxis_title="discharge (cms)")
fig.show()

In [ ]:
summary = run_registry.copy()
for column in [
    "time_to_peak_seconds",
    "fall_time_seconds",
    "fall_time_50_seconds",
    "fall_time_10_seconds",
    "e_folding_time_seconds",
    "lag_to_inflow_peak_seconds",
]:
    summary[column.replace("_seconds", "_hours")] = pd.to_numeric(summary[column], errors="coerce") / 3600.0

px.scatter(
    summary,
    x="time_to_peak_hours",
    y="peak_discharge_cms",
    hover_name="state_id",
    color="peak_attenuation_ratio",
    title="Peak magnitude vs. time to peak",
)